In [49]:
from sc_mbm.mae_for_eeg import MAEforEEG
from dataset import eeg_pretrain_dataset
import matplotlib.pyplot as plt
import numpy as np

from torch.utils.data import DataLoader
import torch
import os

In [50]:
device = torch.device(f'cuda') if torch.cuda.is_available() else torch.device('cpu')
print(device)
print(os.getcwd())
dataset_pretrain = eeg_pretrain_dataset(path='../datasets/eeg_train/')


cpu
/dgxb_home/ext03/eeg-ml/DreamDiffusion/code


In [51]:


print(f'Dataset size: {len(dataset_pretrain)}\n Time len: {dataset_pretrain.data_len}')
sampler = torch.utils.data.DistributedSampler(dataset_pretrain, rank=0) if torch.cuda.device_count() > 1 else None 

dataloader_eeg = DataLoader(dataset_pretrain, batch_size=16, sampler=sampler, 
            shuffle=(sampler is None), pin_memory=True)

model = MAEforEEG(time_len=512, patch_size=16, embed_dim=512,
                    decoder_embed_dim=1024, depth=24, 
                    num_heads=16, decoder_num_heads=16, mlp_ratio=0.8,
                    focus_range=None, focus_rate=0.8, 
                    img_recon_weight=None, use_nature_img_loss=None)  

Dataset size: 6941
 Time len: 512


In [55]:
sample = next(iter(dataloader_eeg))['eeg']
sample = sample.to(device)
loss, pred, mask = model(sample, mask_ratio=0.5)
print(sample.shape, mask.shape)

sample_with_mask = sample.to('cpu').squeeze(0)[0].numpy().reshape(-1, model.patch_size)
print(sample_with_mask.shape)

pred = model.unpatchify(pred).to('cpu').squeeze(0)[0].detach().numpy()

sample = sample.to('cpu').squeeze(0)[0].numpy()
mask = mask.to('cpu').numpy().reshape(-1)
print(loss.detach())

x_axis = np.arange(0, sample.shape[-1])
# plt.plot(x_axis, sample[0])

plt.figure()
# 
# s=0
# print(len(mask))
# for x, m in zip(sample_with_mask,mask):
#             print(len(x))
#             if m == 0:
#                 plt.plot(x_axis[s:s+len(x)], x, color='#1f77b4')
#             s += len(x)
# 
plt.figure()

# plt.plot(x_axis, pred[0])

torch.Size([16, 128, 512]) torch.Size([16, 32])
(4096, 16)
tensor(-0.4296)


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>